### **IMPORTANT NOTE: The runtime hardware type for this notebook must be T4 GPU in order to work with Facer**

In [1]:
# DeepFace imports
!pip install deepface
!pip install --upgrade h5py
from deepface import DeepFace

# Facer imports
!pip install git+https://github.com/FacePerceiver/facer.git@main
import facer
import sys
import torch

# For most common attribute
from collections import Counter

from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.0 MB/s eta 0:00:00
26-05-14 04:39:38 - Directory /root/.deepface has been created
26-05-14 04:39:38 - Directory /root/.deepface/weights has been created
  Cloning https://github.com/FacePerceiver/facer.git (to revision main) to /tmp/pip-req-build-a8edpthl
  Running command git clone --filter=blob:none --quiet https://github.com/FacePerceiver/facer.git /tmp/pip-req-build-a8edpthl
  Resolved https://github.com/FacePerceiver/facer.git to commit ddd35c76ff840174b8a540

In [ ]:
# The file path should direct to the location of the Personas folder
folder_path = Path("/content/drive/MyDrive/Personas")

deepface_analysis = []
facer_analysis = []

# Code to initialize Facer
sys.path.append("..")
device = "cuda" if torch.cuda.is_available() else "cpu"
face_detector = facer.face_detector("retinaface/mobilenet", device=device)
face_attr = facer.face_attr("farl/celeba/224", device=device)

# Iterate through folders in the Personas folder
for folder in folder_path.iterdir():
  if folder.is_dir():
    print(f"Processing directory: {folder}")
    # Separate analyses by post (so attribute values can be averaged later)
    post_analysis = []
    facer_post_analysis = []

    # Iterate through files/subdirectories within the current directory in order
    for file_entry in sorted(folder.iterdir()):
      print(f"Processing file/entry in directory: {file_entry}")

      try:
        # DeepFace analysis of personas
        analysis = DeepFace.analyze(file_entry)
        # Only add dominant traits to list
        dominant_traits = [analysis[0]["dominant_emotion"],
                           analysis[0]["age"],
                           analysis[0]["dominant_gender"],
                           analysis[0]["dominant_race"]]
        post_analysis.append(dominant_traits)

      # Exception in analyzing face, most likely cannot detect face
      # Append null value to the list so the order of the personas are intact
      except:
        post_analysis.append(None)

      try:
        # Facer analysis of personas
        # Taken from example code on Facer's github
        image = facer.hwc2bchw(facer.read_hwc(file_entry)).to(device=device)

        with torch.inference_mode():
            faces = face_detector(image)

        # Shows the analyzed images but commented out for faster processing
        # facer.show_bchw(facer.draw_bchw(image.clone(), faces))
        with torch.inference_mode():
            faces = face_attr(image, faces)
        labels = face_attr.labels
        # Get the "first" face's attributes because it can detect multiple faces in pictures
        # (if theres multiple, but there's only one)
        face1_attrs = faces["attrs"][0]

        # Add the traits that have a 50% or more confidence to ensure accurate results
        persona_analysis = []
        for prob, label in zip(face1_attrs, labels):
          if prob > 0.5:
            persona_analysis.append(label)

        facer_post_analysis.append(persona_analysis)

      except:
        print("No face detected")
        post_analysis.append(None)


    deepface_analysis.append(post_analysis)
    facer_analysis.append(facer_post_analysis)
#     print("post_analysis", post_analysis)
#     print("facer_post_analysis", facer_post_analysis)

# print("deepface_analysis", deepface_analysis)
# print("facer_analysis", facer_analysis)

Downloading: "https://github.com/elliottzheng/face-detection/releases/download/0.0.1/mobilenet0.25_Final.pth" to /root/.cache/torch/hub/checkpoints/mobilenet0.25_Final.pth


100%|██████████| 1.71M/1.71M [00:00<00:00, 86.8MB/s]
Downloading: "https://github.com/FacePerceiver/facer/releases/download/models-v1/face_attribute.farl.celeba.pt" to /root/.cache/torch/hub/checkpoints/face_attribute.farl.celeba.pt
100%|██████████| 327M/327M [00:10<00:00, 33.0MB/s]


Processing directory: /content/drive/MyDrive/663 Data Privacy/Personas/Post 1
Processing file/entry in directory: /content/drive/MyDrive/663 Data Privacy/Personas/Post 1/B.PNG


Action: emotion:   0%|          | 0/4 [00:00<?, ?it/s]

26-05-14 04:40:23 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


...

Action: race: 100%|██████████| 4/4 [00:00<00:00, 40.02it/s]


Processing file/entry in directory: /content/drive/MyDrive/663 Data Privacy/Personas/Post 13/I.PNG


Action: race: 100%|██████████| 4/4 [00:00<00:00, 42.32it/s]


Processing file/entry in directory: /content/drive/MyDrive/663 Data Privacy/Personas/Post 13/J.PNG


Action: race: 100%|██████████| 4/4 [00:00<00:00, 40.07it/s]


In [3]:
print(len(post_analysis))
print(len(facer_post_analysis))
print(len(deepface_analysis))
print(len(facer_analysis ))

deepface_persona_traits = [{"emotion": [], "age": [], "gender": [], "race": []},
                  {"emotion": [], "age": [], "gender": [], "race": []},
                  {"emotion": [], "age": [], "gender": [], "race": []},
                  {"emotion": [], "age": [], "gender": [], "race": []},
                  {"emotion": [], "age": [], "gender": [], "race": []},
                  {"emotion": [], "age": [], "gender": [], "race": []},
                  {"emotion": [], "age": [], "gender": [], "race": []},
                  {"emotion": [], "age": [], "gender": [], "race": []},
                  {"emotion": [], "age": [], "gender": [], "race": []}
                  ]

facer_persona_traits = [[],[],[],[],[],[],[],[],[]]

# Iterate through deepface list and sort traits to their respective persona
for analysis in deepface_analysis:
  index = 0
  for trait in analysis:
    if trait != None:
      deepface_persona_traits[index]["emotion"].append(trait[0])
      deepface_persona_traits[index]["age"].append(trait[1])
      deepface_persona_traits[index]["gender"].append(trait[2])
      deepface_persona_traits[index]["race"].append(trait[3])
    index += 1

# Iterate through facer list and sort traits to their respective persona
for analysis in facer_analysis:
  index = 0
  for traits in analysis:
    if traits != None:
      for trait in traits:
        facer_persona_traits[index].append(trait)
    index += 1

# print(deepface_persona_traits)
# print(len(deepface_persona_traits))
# print(facer_persona_traits)
# print(len(facer_persona_traits))

9
9
13
13


In [4]:
most_common_persona_traits_list = []

# Use counter to keep track of most common outputs for each trait in deepface traits list
for i in range(len(deepface_persona_traits)):
  emotion = Counter(deepface_persona_traits[i]["emotion"])
  age = Counter(deepface_persona_traits[i]["age"])
  gender = Counter(deepface_persona_traits[i]["gender"])
  race = Counter(deepface_persona_traits[i]["race"])

  # Get the most common dominant attribute for each persona
  most_common_persona_traits = [emotion.most_common(1)[0], age.most_common(1)[0], gender.most_common(1)[0], race.most_common(1)[0]]
  most_common_persona_traits_list.append(most_common_persona_traits)

  print("Persona " + str(i+1) + ": " + str(most_common_persona_traits))

# Use counter to count each instance of a trait to see how many times each trait is detected in all photos of the persona
for i in range(len(facer_persona_traits)):
  traits = Counter(facer_persona_traits[i])

  # Get the most common dominant attribute for each persona
  most_common_persona_traits = traits.most_common()
  most_common_persona_traits_list.append(most_common_persona_traits)

  print("Persona " + str(i+1) + ": " + str(most_common_persona_traits))

Persona 1: [('happy', 5), (31, 2), ('Man', 9), ('latino hispanic', 4)]
Persona 2: [('happy', 5), (32, 3), ('Woman', 10), ('latino hispanic', 7)]
Persona 3: [('happy', 4), (32, 4), ('Woman', 9), ('white', 12)]
Persona 4: [('happy', 5), (24, 2), ('Woman', 9), ('white', 12)]
Persona 5: [('happy', 7), (42, 1), ('Man', 12), ('latino hispanic', 5)]
Persona 6: [('happy', 8), (24, 3), ('Woman', 5), ('asian', 8)]
Persona 7: [('sad', 7), (33, 2), ('Woman', 10), ('white', 11)]
Persona 8: [('happy', 5), (28, 4), ('Man', 13), ('white', 9)]
Persona 9: [('happy', 7), (31, 3), ('Woman', 10), ('asian', 6)]
Persona 1: [('No_Beard', 13), ('Bags_Under_Eyes', 12), ('Eyeglasses', 12), ('Gray_Hair', 12), ('Wavy_Hair', 9), ('Big_Nose', 7), ('Smiling', 6), ('High_Cheekbones', 4), ('Male', 4), ('Mouth_Slightly_Open', 4), ('Blurry', 1), ('Young', 1), ('Narrow_Eyes', 1)]
Persona 2: [('No_Beard', 13), ('Wavy_Hair', 13), ('Young', 13), ('Bags_Under_Eyes', 7), ('High_Cheekbones', 7), ('Wearing_Necklace', 6), ('Mouth